# Baseline — Этап 0: инфраструктура

Здесь мы готовим фундамент, на котором будут стоять все ретриверы:

1. **Метрика `MAP@10`** — точно по формуле задачи, с самопроверкой на ручном примере.
2. **Загрузка и очистка данных** — HTML → текст, выделение полей `title` / `body` /
   тексты внутренних ссылок; проверки целостности.
3. **Единый оценочный harness** — функция, которая по предсказаниям
   `{query_id: [article_id, ...]}` считает `MAP@10` на `calibration` и возвращает
   пер-запросные `AP@10` для разбора ошибок.

Код написан так, чтобы устоявшиеся функции можно было без изменений перенести в
`src/` (метрика → `src/metrics.py`, работа с данными → `src/data.py`).

## 0.1 Импорты и конфигурация

In [1]:
import re
import html as html_lib
from pathlib import Path
from collections import Counter
from typing import Iterable, Sequence

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

pd.set_option("display.max_colwidth", 120)

DATA_DIR = Path("../data/candidate_data")
assert DATA_DIR.exists(), f"нет папки с данными: {DATA_DIR.resolve()}"

K = 10  # горизонт метрики MAP@K и максимум статей в ответе (из условия задачи)

## 0.2 Метрика MAP@10

**Average Precision@K** для одного запроса:

$$AP@K = \frac{1}{\min(m, K)} \sum_{i=1}^{K} P(i)\cdot rel(i)$$

где
- $m$ — число правильных статей (`ground_truth`) для запроса,
- $rel(i)=1$, если документ на позиции $i$ релевантен, иначе $0$,
- $P(i)$ — точность на отсечке $i$ = (число релевантных среди первых $i$) / $i$.

Нормировка на $\min(m, K)$ — стандартная для соревновательного MAP@K (у нас
$m \le 4 < K=10$, поэтому фактически делим на $m$). Идея метрики: вклад даёт
только релевантный документ, и тем больший, чем выше он стоит и чем плотнее
релевантные идут вверху списка.

**MAP@K** — это $AP@K$, усреднённый по всем запросам.

In [2]:
def average_precision_at_k(predicted: Sequence[int],
                           relevant: Iterable[int],
                           k: int = K) -> float:
    """AP@k для одного запроса.

    predicted — ранжированный список article_id (от самого релевантного).
    relevant  — множество правильных article_id (ground_truth).
    Дубликаты в predicted игнорируются (по условию их и не должно быть).
    """
    relevant = set(relevant)
    if not relevant:
        return 0.0

    # убираем повторы, сохраняя порядок первого вхождения: повтор не должен
    # ни добивать скор дважды, ни «съедать» позицию у следующего документа
    seen, deduped = set(), []
    for doc in predicted:
        if doc not in seen:
            seen.add(doc)
            deduped.append(doc)

    hits = 0            # сколько релевантных уже встретили
    score = 0.0         # накопленная сумма P(i)*rel(i)
    for i, doc in enumerate(deduped[:k], start=1):
        if doc in relevant:
            hits += 1
            score += hits / i          # это и есть P(i) в момент попадания
    return score / min(len(relevant), k)


def mean_average_precision_at_k(predictions: dict[int, Sequence[int]],
                                truth: dict[int, Iterable[int]],
                                k: int = K) -> float:
    """MAP@k — среднее AP@k по всем запросам из truth."""
    aps = [average_precision_at_k(predictions.get(qid, []), rel, k)
           for qid, rel in truth.items()]
    return float(np.mean(aps)) if aps else 0.0

**Самопроверка метрики** на ручном примере — чтобы формула точно совпадала с ожиданием.

In [3]:
# gt = {10, 20}. Кандидаты: [10, 99, 20, ...]
#   поз.1: 10 релевантна -> P=1/1=1.0
#   поз.2: 99 нет
#   поз.3: 20 релевантна -> P=2/3
#   AP = (1.0 + 2/3) / min(2,10) = 1.6667 / 2 = 0.8333
ap = average_precision_at_k([10, 99, 20, 5, 6], {10, 20})
assert abs(ap - 0.8333333) < 1e-6, ap

# идеальный порядок -> AP = 1.0
assert abs(average_precision_at_k([10, 20, 99], {10, 20}) - 1.0) < 1e-9
# ни одного попадания -> 0.0
assert average_precision_at_k([1, 2, 3], {10, 20}) == 0.0
# один релевантный на 2-й позиции -> (1/2)/1 = 0.5
assert abs(average_precision_at_k([9, 10, 11], {10}) - 0.5) < 1e-9
# повтор не должен «добивать» скор дважды
assert abs(average_precision_at_k([10, 10, 20], {10, 20}) - average_precision_at_k([10, 20], {10, 20})) < 1e-9

print("OK: метрика проходит все самопроверки")
print("пример AP@10 =", round(ap, 4))

OK: метрика проходит все самопроверки
пример AP@10 = 0.8333


## 0.3 Загрузка и очистка данных

Тела статей — сырой HTML со служебной разметкой (табы, спойлеры, таблицы,
factoid-блоки, картинки). Для поиска нам нужен чистый текст. Дополнительно
вытаскиваем **тексты внутренних ссылок** (anchor'ы) — на этапе EDA мы убедились,
что это перефразы-синонимы заголовков, полезный сигнал для матчинга.

In [4]:
def html_to_text(raw: str) -> str:
    """Грубая, но устойчивая очистка HTML статьи в plain-текст.

    - выкидываем script/style;
    - <img> заменяем его alt-текстом (там бывают осмысленные подписи);
    - получаем текст с пробелами-разделителями и схлопываем пробелы.
    """
    if not isinstance(raw, str) or not raw:
        return ""
    soup = BeautifulSoup(raw, "lxml")
    for tag in soup(["script", "style"]):
        tag.decompose()
    for img in soup.find_all("img"):
        img.replace_with(" " + (img.get("alt") or "") + " ")
    text = soup.get_text(separator=" ")
    text = html_lib.unescape(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


_LINK_RE = re.compile(r"support\.avito\.ru/articles/(\d+)")


def extract_anchor_text(raw: str) -> str:
    """Собирает тексты всех внутренних ссылок на статьи справки в одну строку."""
    if not isinstance(raw, str) or not raw:
        return ""
    soup = BeautifulSoup(raw, "lxml")
    parts = []
    for a in soup.find_all("a", href=True):
        if _LINK_RE.search(a["href"]):
            t = a.get_text(" ", strip=True)
            if t:
                parts.append(t)
    return " ".join(parts)


def extract_linked_ids(raw: str) -> list[int]:
    """Список article_id, на которые ссылается статья (для графа ссылок)."""
    if not isinstance(raw, str):
        return []
    return [int(x) for x in _LINK_RE.findall(raw)]

In [5]:
def load_articles() -> pd.DataFrame:
    """Читает articles.f и добавляет очищенные текстовые поля."""
    df = pd.read_feather(DATA_DIR / "articles.f").copy()
    df["title"] = df["title"].fillna("").astype(str)
    df["body_text"] = df["body"].map(html_to_text)
    df["anchors"] = df["body"].map(extract_anchor_text)
    df["linked_ids"] = df["body"].map(extract_linked_ids)
    df["n_words"] = df["body_text"].str.count(r"\w+")
    return df


def parse_gt(s: str) -> list[int]:
    """'1909 4396' -> [1909, 4396]."""
    if not isinstance(s, str):
        return []
    return [int(x) for x in s.split()]


def load_calibration() -> pd.DataFrame:
    df = pd.read_feather(DATA_DIR / "calibration.f").copy()
    df["query_text"] = df["query_text"].fillna("").astype(str)
    df["gt_ids"] = df["ground_truth"].map(parse_gt)
    return df


def load_test() -> pd.DataFrame:
    df = pd.read_feather(DATA_DIR / "test.f").copy()
    df["query_text"] = df["query_text"].fillna("").astype(str)
    return df


articles = load_articles()
calibration = load_calibration()
test = load_test()

print("articles   ", articles.shape)
print("calibration", calibration.shape)
print("test       ", test.shape)
articles[["article_id", "title", "body_text", "anchors", "n_words"]].head(3)

articles    (793, 7)
calibration (500, 4)
test        (500, 2)


,article_id,title,body_text,anchors,n_words
0,1730,Имя или название компании,Зайдите в раздел Управление профилем . Нажмите на карандашик в правом верхнем углу. Введите новое имя или название и...,профессиональный профиль,0
1,1746,"Понять, что профиль заблокирован","Проверьте, какое сообщение вы видите при входе в профиль. Профиль заблокирован Доступ к профилю ограничен Доступ огр...",причины блокировки Ваши объявления могли вводить покупателей в заблуждение У вас есть другой профиль с пройденной пр...,0
2,1747,Не допустить блокировки профиля,"Не заводите несколько аккаунтов для продаж в одной категории. Так поступают пользователи, которые хотят обойти наши ...",Ваши объявления могли вводить покупателей в заблуждение У вас есть другой профиль с пройденной проверкой Мы заметили...,0


**Проверки целостности** — то, что нам гарантирует условие, лучше подтвердить в коде.

In [6]:
valid_ids = set(articles["article_id"])

# 1. article_id уникальны
assert articles["article_id"].is_unique, "article_id не уникальны!"

# 2. все ground_truth-id существуют в корпусе
gt_all = {i for ids in calibration["gt_ids"] for i in ids}
missing = gt_all - valid_ids
assert not missing, f"ground_truth ссылается на отсутствующие статьи: {missing}"

# 3. нет пустых очищенных тел (иначе статью нечем матчить)
empty_bodies = int((articles["body_text"].str.len() == 0).sum())

# 4. в тесте нет пропусков query_id / query_text
assert test["query_id"].notna().all() and test["query_text"].str.len().gt(0).all()

print("OK целостность:")
print(f"  статей: {len(articles)}, уникальных id: {articles['article_id'].nunique()}")
print(f"  правильных статей в calibration: {len(gt_all)} (все существуют)")
print(f"  статей с пустым body_text: {empty_bodies}")
print(f"  запросов: calibration={len(calibration)}, test={len(test)}")

OK целостность:
  статей: 793, уникальных id: 793
  правильных статей в calibration: 79 (все существуют)
  статей с пустым body_text: 0
  запросов: calibration=500, test=500


## 0.4 Оценочный harness

Единая точка для замера любого ретривера на `calibration`. Ретривер отдаёт
предсказания в виде `{query_id: [article_id, ...]}` — harness считает `MAP@10`
и, для разбора ошибок, возвращает пер-запросные `AP@10`.

In [7]:
def build_truth(cal: pd.DataFrame) -> dict[int, set[int]]:
    return {int(q): set(ids) for q, ids in zip(cal["query_id"], cal["gt_ids"])}


def evaluate(predictions: dict[int, Sequence[int]],
             cal: pd.DataFrame = calibration,
             k: int = K) -> tuple[float, pd.DataFrame]:
    """Возвращает (MAP@k, таблица пер-запросных AP@k, отсортированная по возрастанию).

    Таблица удобна для error-analysis: сразу видно, на каких запросах провал.
    """
    truth = build_truth(cal)
    rows = []
    for _, r in cal.iterrows():
        qid = int(r["query_id"])
        ap = average_precision_at_k(predictions.get(qid, []), truth[qid], k)
        rows.append({"query_id": qid, "query_text": r["query_text"],
                     "n_gt": len(truth[qid]), "ap": ap})
    per_query = pd.DataFrame(rows).sort_values("ap").reset_index(drop=True)
    return float(per_query["ap"].mean()), per_query

**Санити-чек harness'а на «глупых» бейзлайнах** — убеждаемся, что цифры осмысленны.

- *Оракул* (возвращаем сам `ground_truth`) обязан дать `MAP@10 = 1.0`.
- *Популярность* (всем один и тот же топ самых частых статей из calibration) —
  слабый, но ненулевой ориентир снизу. Это тот же приём, что в примере из условия
  задачи, и он полезен как нижняя граница.

In [8]:
# Оракул: идеальные предсказания
truth = build_truth(calibration)
oracle_pred = {qid: list(rel) for qid, rel in truth.items()}
map_oracle, _ = evaluate(oracle_pred)
print(f"MAP@10 оракула (sanity=1.0):     {map_oracle:.4f}")
assert abs(map_oracle - 1.0) < 1e-9

# Популярность: топ-10 самых частых ground_truth-статей всем одинаково
top_popular = [aid for aid, _ in Counter(
    i for ids in calibration["gt_ids"] for i in ids).most_common(K)]
pop_pred = {int(q): top_popular for q in calibration["query_id"]}
map_pop, per_query_pop = evaluate(pop_pred)
print(f"MAP@10 популярности (нижняя грань): {map_pop:.4f}")
print("топ-популярных id:", top_popular)

MAP@10 оракула (sanity=1.0):     1.0000
MAP@10 популярности (нижняя грань): 0.3176
топ-популярных id: [4219, 1951, 4234, 2646, 4328, 4400, 4214, 4384, 4440, 4387]


## Итог Этапа 0

Готово и проверено:

- **`average_precision_at_k` / `mean_average_precision_at_k`** — метрика с
  самопроверкой (→ `src/metrics.py`).
- **`load_articles / load_calibration / load_test`** + `html_to_text`,
  `extract_anchor_text`, `extract_linked_ids` — данные и очистка (→ `src/data.py`).
- **`evaluate`** — harness с пер-запросным разбором и осмысленными нижней/верхней
  границами (популярность / оракул).

Дальше (Этап 1) на этом фундаменте строим лексический BM25-baseline и меряем его
этим же `evaluate`.

In [9]:
# === STAGE 1 START ===
# Этап 1: лексический baseline (BM25). Ячейка-маркер для идемпотентного пере-запуска билдера.

---
# Этап 1 — Лексический baseline (BM25)

Задача этапа: получить **прозрачную опорную цифру** MAP@10 чисто лексическим
методом и разобрать его ошибки. BM25 (Okapi) — стандарт для лексического поиска:
он вознаграждает совпадение редких терминов запроса со статьёй и штрафует за
длину документа, чтобы длинные статьи не «перевешивали» короткие.

Всё считаем на `scikit-learn` + `scipy` (уже в базовом окружении), без внешних
BM25-библиотек — так виден каждый шаг формулы. Русскую морфологию гасим
Snowball-стеммером (`доставка/доставки/доставку → доставк`).

## 1.1 Предобработка текста

Пользователи пишут разговорно, с разными словоформами. Чтобы лексический матч
не разваливался на «доставка» vs «доставки», приводим текст к нормальной форме:

1. нижний регистр + `ё → е` (устраняем разнобой написания);
2. токенизация по буквам/цифрам (кириллица+латиница), выкидываем токены короче 2;
3. убираем **функциональные** стоп-слова (предлоги, местоимения) — но не трогаем
   доменные слова;
4. стемминг Snowball — схлопываем словоформы к общей основе.

In [10]:
import snowballstemmer

_stemmer = snowballstemmer.stemmer("russian")
_TOKEN_RE = re.compile(r"[а-яёa-z0-9]+")

# компактный список русских функциональных стоп-слов (предлоги, союзы,
# местоимения, частицы). Доменные слова (доставка, возврат, профиль) НЕ включаем.
RU_STOPWORDS = set("""
и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по
только ее мне было вот от меня еще нет о из ему теперь когда даже ну вдруг ли если
уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей
может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз
тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом
один почти мой тем чтобы нее сейчас были куда зачем всех можно при наконец два об
другой хоть после над больше тот через эти нас про всего них какая много разве три
эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более
всегда конечно всю между это мне при да нам ваш наш свой ещё также этих том эта
""".split())


def tokenize(text: str, stem: bool = True, drop_stop: bool = True,
             min_len: int = 2) -> list[str]:
    """Текст -> список нормализованных токенов."""
    if not isinstance(text, str):
        return []
    text = text.lower().replace("ё", "е")
    toks = _TOKEN_RE.findall(text)
    toks = [t for t in toks if len(t) >= min_len]
    if drop_stop:
        toks = [t for t in toks if t not in RU_STOPWORDS]
    if stem:
        toks = _stemmer.stemWords(toks)
    return toks


# демонстрация на реальном запросе
demo = "Здравствуйте,подскажите как мне отправить кроссовки покупателю через пункт «Авито»?"
print("исходный запрос:", demo)
print("токены:", tokenize(demo))

исходный запрос: Здравствуйте,подскажите как мне отправить кроссовки покупателю через пункт «Авито»?
токены: ['здравств', 'подскаж', 'отправ', 'кроссовк', 'покупател', 'пункт', 'авит']


## 1.2 BM25 «с нуля»

Формула Okapi BM25 для документа $d$ и запроса $Q$:

$$score(d, Q) = \sum_{t \in Q} IDF(t)\cdot \frac{f(t,d)\,(k_1+1)}{f(t,d) + k_1\,(1 - b + b\,\frac{|d|}{avgdl})}$$

- $f(t,d)$ — частота терма $t$ в документе;
- $IDF(t) = \ln\!\left(1 + \frac{N - df(t) + 0.5}{df(t) + 0.5}\right)$ — редкие термы важнее;
- $|d|/avgdl$ — нормировка на длину документа (параметр $b$);
- $k_1$ — насыщение по частоте терма.

Реализуем на разреженных матрицах `scipy`: терм-документная матрица счётчиков
строится `CountVectorizer` поверх уже токенизированного текста.

In [11]:
from sklearn.feature_extraction.text import CountVectorizer


class BM25:
    """Классический Okapi BM25 на разреженной терм-документной матрице."""

    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1, self.b = k1, b

    def fit(self, tokenized_docs: list[list[str]]):
        # analyzer=identity: на вход уже поданы списки токенов, не строки
        self.vec = CountVectorizer(analyzer=lambda t: t, min_df=1)
        X = self.vec.fit_transform(tokenized_docs)     # (N документов x V термов), счётчики
        self.vocab = self.vec.vocabulary_              # term -> column index
        self.X = X.tocsc()                             # CSC — быстрый доступ по колонкам (термам)

        N, V = X.shape
        df = np.asarray((X > 0).sum(axis=0)).ravel()   # в скольких документах встречается терм
        self.idf = np.log(1 + (N - df + 0.5) / (df + 0.5))
        dl = np.asarray(X.sum(axis=1)).ravel()         # длина каждого документа (в токенах)
        avgdl = dl.mean()
        # знаменатель BM25 без f(t,d): B_d = k1*(1 - b + b*|d|/avgdl), считаем один раз
        self.B_d = self.k1 * (1 - self.b + self.b * dl / avgdl)
        self.N = N
        return self

    def get_scores(self, query_tokens: list[str]) -> np.ndarray:
        """BM25-скор каждого документа против запроса (вектор длины N)."""
        scores = np.zeros(self.N)
        for t in set(query_tokens):                    # каждый терм запроса учитываем один раз
            j = self.vocab.get(t)
            if j is None:                              # терма нет в корпусе — пропускаем
                continue
            col = self.X[:, j]                         # разреженная колонка терма
            rows, f = col.indices, col.data.astype(float)
            # вклад терма только в документы, где он встречается
            scores[rows] += self.idf[j] * (f * (self.k1 + 1)) / (f + self.B_d[rows])
        return scores

## 1.3 Сборка документов и раннер

Каждую статью представляем как «документ» из нескольких полей. Заголовок (`title`)
— короткий, но очень точный сигнал, поэтому даём ему вес через повтор токенов
(в BM25 это увеличивает $f(t,d)$ для слов заголовка). Тело и anchor-тексты
добавляем с весом 1. Все веса — гиперпараметры, которые сравним ниже.

In [12]:
# выравниваем порядок: строка матрицы i  <->  article_id
ARTICLE_IDS = articles["article_id"].to_numpy()


def build_doc_tokens(row, w_title=2, w_body=1, w_anchor=1, **tok_kw) -> list[str]:
    """Собирает токены статьи с полевыми весами (вес = повтор поля)."""
    toks = []
    toks += tokenize(row["title"], **tok_kw) * w_title
    toks += tokenize(row["body_text"], **tok_kw) * w_body
    if w_anchor:
        toks += tokenize(row["anchors"], **tok_kw) * w_anchor
    return toks


def run_bm25(w_title=2, w_body=1, w_anchor=1, k1=1.5, b=0.75,
             queries=calibration, stem=True, drop_stop=True):
    """Полный прогон BM25 -> (MAP@10, per_query, обученная модель, предсказания)."""
    tok_kw = dict(stem=stem, drop_stop=drop_stop)
    docs = [build_doc_tokens(r, w_title, w_body, w_anchor, **tok_kw)
            for _, r in articles.iterrows()]
    bm25 = BM25(k1=k1, b=b).fit(docs)

    preds = {}
    for _, r in queries.iterrows():
        q = tokenize(r["query_text"], **tok_kw)
        scores = bm25.get_scores(q)
        top = np.argpartition(-scores, K)[:K]          # топ-K без полной сортировки
        top = top[np.argsort(-scores[top])]            # и упорядочим эти K
        preds[int(r["query_id"])] = [int(ARTICLE_IDS[i]) for i in top]

    map_score, per_query = evaluate(preds, queries)
    return map_score, per_query, bm25, preds


# основной прогон: title(x2) + body + anchors, стемминг вкл.
map_main, per_query_main, bm25_main, preds_main = run_bm25()
print(f"BM25 (title x2 + body + anchors, stem): MAP@10 = {map_main:.4f}")
print(f"  для сравнения — популярность: {map_pop:.4f}, оракул: {map_oracle:.4f}")

BM25 (title x2 + body + anchors, stem): MAP@10 = 0.2718
  для сравнения — популярность: 0.3176, оракул: 1.0000


## 1.4 Абляции — что реально помогает

Меняем по одному фактору и смотрим на MAP@10, чтобы решения были обоснованы
замером, а не интуицией. Помним про смещённость `calibration`: большие различия
значимы, микроскопические — нет.

In [13]:
ablations = [
    ("body только",                dict(w_title=0, w_body=1, w_anchor=0)),
    ("title + body",               dict(w_title=1, w_body=1, w_anchor=0)),
    ("title x2 + body",            dict(w_title=2, w_body=1, w_anchor=0)),
    ("title x3 + body",            dict(w_title=3, w_body=1, w_anchor=0)),
    ("title x2 + body + anchors",  dict(w_title=2, w_body=1, w_anchor=1)),
    ("title x2 + body + anchors x2", dict(w_title=2, w_body=1, w_anchor=2)),
    ("+ без стемминга",            dict(w_title=2, w_body=1, w_anchor=1, stem=False)),
    ("+ без стоп-слов-фильтра",     dict(w_title=2, w_body=1, w_anchor=1, drop_stop=False)),
]
rows = []
for name, kw in ablations:
    m, _, _, _ = run_bm25(**kw)
    rows.append({"конфигурация": name, "MAP@10": round(m, 4)})
abl = pd.DataFrame(rows)
abl

,конфигурация,MAP@10
0,body только,0.2683
1,title + body,0.2723
2,title x2 + body,0.2748
3,title x3 + body,0.2749
4,title x2 + body + anchors,0.2718
5,title x2 + body + anchors x2,0.2702
6,+ без стемминга,0.3038
7,+ без стоп-слов-фильтра,0.2779


**Мини-подбор `k1`, `b`** вокруг стандартных значений (осторожно, чтобы не переобучиться).

In [14]:
grid = []
for k1 in (0.9, 1.2, 1.5, 2.0):
    for b in (0.4, 0.6, 0.75, 0.9):
        m, _, _, _ = run_bm25(w_title=2, w_body=1, w_anchor=1, k1=k1, b=b)
        grid.append({"k1": k1, "b": b, "MAP@10": round(m, 4)})
grid = pd.DataFrame(grid).sort_values("MAP@10", ascending=False)
print("лучшие 5 комбинаций k1/b:")
print(grid.head(5).to_string(index=False))
print("\nдефолт k1=1.5,b=0.75:", grid[(grid.k1==1.5)&(grid.b==0.75)]["MAP@10"].values[0])

лучшие 5 комбинаций k1/b:
 k1    b  MAP@10
2.0 0.40  0.2959
2.0 0.60  0.2950
2.0 0.75  0.2841
1.5 0.40  0.2798
1.5 0.60  0.2754

дефолт k1=1.5,b=0.75: 0.2718


## 1.5 Разбор ошибок

Смотрим запросы с самым низким AP@10: где BM25 промахивается и почему. Это
подсказывает, что должна дотянуть семантика на Этапе 2 (несовпадение лексики
запроса и статьи).

In [15]:
id2title = dict(zip(articles["article_id"], articles["title"]))

worst = per_query_main.head(12)
print("Запросы с худшим AP@10 (BM25):\n")
for _, r in worst.iterrows():
    qid = r["query_id"]
    gt = build_truth(calibration)[qid]
    got = preds_main[qid][:5]
    print(f"AP={r['ap']:.3f} | [{qid}] {r['query_text'][:80]}")
    print("   нужно :", [f"{g}:{id2title.get(g,'?')[:30]}" for g in gt])
    print("   BM25  :", [f"{g}:{id2title.get(g,'?')[:30]}" for g in got[:3]])
    print()

Запросы с худшим AP@10 (BM25):

AP=0.000 | [498] Почему нету доставки за <MONEY>?
   нужно : ['4234:Как продавать и покупать с дос', '1951:Кто оплачивает доставку и скол']
   BM25  : ['3077:Балансы в кошельке Авито: како', '4384:Баланс для покупок', '3470:Вернуть деньги из кошелька Ави']

AP=0.000 | [18] Здравствуйте, почему сумма доставки меняется при оформлении?
   нужно : ['1951:Кто оплачивает доставку и скол']
   BM25  : ['4214:Скидки, бонусы и промокоды', '2962:Заказ в рассрочку', '2646:Оплата заказов с доставкой']

AP=0.000 | [24] №<ID> нужно поменять карту на которую приедут деньги та заблокировано
   нужно : ['4361:Продавцу']
   BM25  : ['4308:Заказать товар с доставкой', '4446:Покупайте только с Авито Доста', '2721:Как устроен процесс бронирован']

AP=0.000 | [26] Добрый вечер. Покупатель задает вопрос. Типа не настроено что-то для крупной пос
   нужно : ['4234:Как продавать и покупать с дос']
   BM25  : ['2295:Благотворительные программы', '4346:Сохранённый поиск', '1837:Отве

In [16]:
# сколько запросов вообще «безнадёжны» для BM25: ни одной правильной статьи в топ-10
truth = build_truth(calibration)
zero = sum(1 for qid in truth
           if not (set(preds_main[qid]) & truth[qid]))
recall_at10 = np.mean([len(set(preds_main[qid]) & truth[qid]) / len(truth[qid])
                       for qid in truth])
print(f"Запросов без единого попадания в топ-10: {zero} / {len(truth)} ({zero/len(truth)*100:.1f}%)")
print(f"Средний recall@10 (доля найденных правильных статей): {recall_at10:.3f}")
print("\nРаспределение AP@10:")
print(per_query_main["ap"].describe().round(3))

Запросов без единого попадания в топ-10: 171 / 500 (34.2%)
Средний recall@10 (доля найденных правильных статей): 0.573

Распределение AP@10:
count    500.000
mean       0.272
std        0.328
min        0.000
25%        0.000
50%        0.143
75%        0.417
max        1.000
Name: ap, dtype: float64


## Итог Этапа 1

- Реализован прозрачный **BM25** с русской предобработкой (стемминг + стоп-слова),
  полевым взвешиванием (`title`/`body`/`anchors`) и мини-подбором `k1`, `b`.
- Абляции показывают вклад каждого поля и предобработки; выбранная конфигурация —
  опорная лексическая точка для сравнения с семантикой.
- Разбор ошибок выделил запросы, где лексического совпадения не хватает
  (перефразы, синонимы) — это цель для Этапа 2 (плотные эмбеддинги / гибрид).

Числа и худшие запросы — в выводах ячеек выше.

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-reranker-v2-m3")

sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium."
]
embeddings = model.encode(sentences)

similarities = model.similarity(embeddings, embeddings)
print(similarities.shape)

No sentence-transformers model found with name BAAI/bge-reranker-v2-m3. Creating a new one with MEAN pooling.


model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

c:\Users\ilyak\miniconda3\envs\torch_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ilyak\.cache\huggingface\hub\models--BAAI--bge-reranker-v2-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some weights of XLMRobertaModel were not initialized from the model checkpoint at BAAI/bge-reranker-v2-m

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

AttributeError: 'SentenceTransformer' object has no attribute 'similarity'